# Topic 38 — BERT-style Models
### Theory → tokenizer internals → load pretrained DistilBERT → fine-tune on your cyberbullying-style task.

Everything from Topic 37 (self-attention, positional info, layer norm, feed-forward blocks) is the
architecture INSIDE BERT. What makes BERT special is *how it was trained*:

- **Pretraining**: BERT is first trained on a MASSIVE unlabeled text corpus (Wikipedia, books) using
  **Masked Language Modeling (MLM)** — randomly hide ~15% of tokens and train the model to predict
  them from context. Because BERT sees context from BOTH directions (left AND right of the masked
  word) simultaneously, it's called **bidirectional**.
- **Fine-tuning**: you then take that pretrained model and continue training it (usually much less
  data, much fewer steps) on YOUR specific labeled task — e.g. cyberbullying classification. The
  model already "understands" language; fine-tuning just teaches it your task.

This transfer-learning approach is why BERT-family models usually beat classical ML (Topic 25) or
training an LSTM from scratch (Topic 34), especially on small labeled datasets like a typical
academic cyberbullying dataset.

In [ ]:
!pip install transformers datasets -q
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Tokenizers — subword tokenization

Unlike the word-level tokenization from Topic 21/34, BERT-family models use **subword
tokenization** (WordPiece/BPE): common words stay whole, rare words get split into meaningful
pieces. This handles typos, slang, and rare words far more gracefully than a fixed word vocabulary.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

text = "you are stupidddd and worthlesssss"   # deliberately misspelled/elongated (Topic 21's concern)
tokens = tokenizer.tokenize(text)
print("subword tokens:", tokens)
# Notice unusual words get split into recognizable sub-pieces (often marked with '##') --
# far more graceful than an out-of-vocabulary word being dropped entirely (Topic 22's CountVectorizer).

In [ ]:
encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print("\ndecoded back to text:", tokenizer.decode(encoded["input_ids"][0]))
# input_ids: the token indices (like Topic 34's vocab indices, but BERT's own subword vocab)
# attention_mask: 1 = real token, 0 = padding -- tells the model to IGNORE padded positions
#                 in its attention computation (directly related to Topic 37's masking idea).

## 2. Token embeddings + positional embeddings + attention masks

BERT's input representation for each token = **token embedding** (what the word means) +
**positional embedding** (learned, unlike Topic 37's fixed sine/cosine) + a **segment embedding**
(which sentence, for tasks with 2 input sentences). The **attention mask** then ensures padding
tokens don't influence real tokens during self-attention.

In [ ]:
model_base = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

with torch.no_grad():
    output = model_base(**{k: v.to(device) for k, v in encoded.items()})

print("last_hidden_state shape:", output.last_hidden_state.shape)
print("-> (batch, seq_len, hidden_dim=768) -- one 768-dim contextual embedding PER TOKEN")
# Each token's final vector already encodes its full bidirectional context -- this is what
# fine-tuning builds on top of.

## 3. The classification head

For classification, a small extra layer (the **classification head**) sits on top of BERT's
output — typically using the `[CLS]` token's final embedding (a special token BERT prepends to
every input, trained to summarize the whole sequence) fed into a `Linear` layer.

In [ ]:
cls_embedding = output.last_hidden_state[:, 0, :]   # the [CLS] token is always position 0
print("[CLS] embedding shape:", cls_embedding.shape)

classification_head = nn.Linear(768, 2).to(device)   # 768 -> 2 classes (bullying / not)
logits = classification_head(cls_embedding)
print("logits:", logits)
# Right now this head is UNTRAINED (random weights) -- fine-tuning trains exactly this kind of
# head (often together with lightly updating BERT's own weights too).

## 4. Fine-tuning DistilBERT on your (toy) cyberbullying task

`AutoModelForSequenceClassification` bundles a pretrained BERT-family model with a classification
head already attached, ready for fine-tuning end-to-end.

In [ ]:
bullying_examples = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "you should just disappear", "you are pathetic",
    "everyone thinks you're an idiot", "just go away nobody likes you",
    "you're so ugly and useless", "why do you even exist",
    "you deserve to be alone", "stop talking you sound stupid",
]
not_bullying_examples = [
    "great job today team", "have a wonderful day", "nice work everyone",
    "thanks for your help", "well done on the project", "excellent effort today",
    "looking forward to the weekend", "congratulations on your achievement",
    "the weather is nice today", "let's grab coffee sometime",
    "i really appreciate your feedback", "the meeting went smoothly",
]
texts = bullying_examples + not_bullying_examples
labels = [1]*len(bullying_examples) + [0]*len(not_bullying_examples)

X_train_txt, X_test_txt, y_train_lbl, y_test_lbl = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

class BertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length",
                                     max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = BertDataset(X_train_txt, y_train_lbl, tokenizer)
test_dataset = BertDataset(X_test_txt, y_test_lbl, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)   # small LR -- we're barely nudging huge pretrained weights

n_epochs = 4   # BERT fine-tuning needs FAR fewer epochs than training from scratch (Topics 30, 34)
losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_losses = []
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)          # HuggingFace models compute loss internally when labels are passed
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    losses.append(np.mean(epoch_losses))
    print(f"epoch {epoch+1}: loss={losses[-1]:.4f}")

plt.figure(figsize=(5, 4))
plt.plot(losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.title("DistilBERT fine-tuning loss")
plt.show()

## 5. Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        labels_batch = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = outputs.logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels_batch.tolist())

print(classification_report(all_labels, all_preds, target_names=["not_bullying", "bullying"], zero_division=0))
# Compare this to Topic 25's classical baselines and Topic 34's LSTM on the SAME toy sentences --
# with such a tiny dataset, differences may be small, but on a real dataset with thousands of
# examples BERT-family models typically pull ahead noticeably.

## 6. Inference on new sentences

In [ ]:
def predict_bert(sentence, model, tokenizer):
    model.eval()
    encoded = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=1)[0]
    pred = probs.argmax().item()
    return ("bullying" if pred == 1 else "not bullying"), probs.cpu().numpy()

for sentence in ["you are worthless and stupid", "have an amazing day everyone", "you're such a loser honestly"]:
    label, probs = predict_bert(sentence, model, tokenizer)
    print(f"'{sentence}' -> {label}  (not_bullying={probs[0]:.2f}, bullying={probs[1]:.2f})")

## 7. BERT / DistilBERT / RoBERTa — which to reach for

- **BERT**: the original — bidirectional, MLM-pretrained. Good general baseline.
- **DistilBERT**: a smaller, faster, "distilled" version of BERT (~40% smaller, ~60% faster, ~97%
  of BERT's performance retained) — a great default for a student project with limited compute,
  used throughout this notebook.
- **RoBERTa**: BERT trained longer, on more data, with some training details tuned — often a bit
  stronger than BERT, at similar size/cost. Worth trying if DistilBERT's results aren't strong enough.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Add 10+ of your own real-style examples to bullying_examples/not_bullying_examples and
#    re-run fine-tuning -- compare classification_report to before.
# 2. Swap "distilbert-base-uncased" for "roberta-base" (both tokenizer AND model) and compare results.
# 3. Try freezing BERT's own weights (for param in model.distilbert.parameters(): param.requires_grad
#    = False) and training ONLY the classification head -- does it still work reasonably, and is it faster?
# 4. Once you have your real cyberbullying dataset, replace `texts`/`labels` here directly --
#    this notebook's fine-tuning code should work with minimal changes, though you'll likely want
#    more epochs and a validation set (Topic 6) for a real experiment.

---
### Next up: **Topic 39 — Hyperparameter Tuning** (GridSearchCV, RandomizedSearchCV).

Say "next" when you're ready.